In [1]:
# A language model is just a probability table over next words. Here's a hand-made one:
next_word = {
    "the": {"morning": 0.30, "train": 0.20, "park": 0.15, "city": 0.10, "weekend": 0.08},
    "good": {"book": 0.5, "coffee": 0.5},
}
def suggest(context, table, k=3):
    ranked = sorted(table.get(context, {}).items(), key=lambda kv: -kv[1])
    return ranked[:k]

print("after 'the' :", suggest("the", next_word))
print("after 'good':", suggest("good", next_word))

after 'the' : [('morning', 0.3), ('train', 0.2), ('park', 0.15)]
after 'good': [('book', 0.5), ('coffee', 0.5)]


In [2]:
CORPUS = '''I drink coffee every morning before work.
The morning train to the city is often late.
She reads a book on the train to work.
We drink tea in the afternoon at the office.
The weather is cold and grey this morning.
He walks to the station in the rain.
On the weekend we walk in the park.
The park is quiet and green in the morning.
I read a good book on the weekend.
Coffee in the morning helps me wake up.
The city is busy and loud during the week.
She takes the late train home from work.
We cook dinner together on the weekend.
The rain falls softly on the quiet street.
He drinks his coffee black and strong.
The office is warm and bright in the winter.
I walk to work when the weather is nice.
The train was late again this morning.
She writes in her journal every evening.
We watch a film together on friday night.
The dog runs happily in the green park.
Morning coffee and a good book make me happy.
The weekend is short but the week is long.
He reads the news on his phone every morning.
The garden is full of flowers in the spring.
I like a quiet morning with hot coffee.
The children play in the park after school.
She cooks a warm dinner on cold evenings.
We travel to the coast in the summer.
The sea is calm and blue in the morning.
Rainy mornings make me want more coffee.
The old cafe on the corner makes good coffee.
He waits for the train on the cold platform.
I finish work and walk home in the evening.
The city lights are bright at night.
She plays the piano softly in the evening.
We read together by the warm fire in winter.
The bakery smells of fresh bread every morning.
I buy fresh bread and coffee on the weekend.
The bus is slower than the morning train.
He enjoys a long walk on sunny weekends.
The library is quiet and full of old books.
She borrows a new book from the library.
Cold weather makes the morning coffee taste better.
We plan a short trip for the long weekend.
The mountains are beautiful in the early morning.
I wake up early and make strong coffee.
The evening is calm after a busy week.
He cooks pasta for dinner on friday.
The rain stopped and the sun came out.
We sit in the park and read our books.
Morning trains are crowded during the week.
She drinks green tea instead of coffee.
The winter mornings are dark and cold.
I love the smell of coffee in the morning.
The weekend market sells fresh fruit and bread.
He reads a chapter of his book before bed.
The quiet street is empty late at night.
We walk the dog in the park every evening.
Good coffee and a good book start my day.'''

with open("corpus.txt", "w") as f:
    f.write(CORPUS)
print("saved corpus.txt —", len(CORPUS), "characters")

saved corpus.txt — 2528 characters


In [3]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

sents = to_sentences(CORPUS)
tokens = [t for s in sents for t in to_tokens(s)]
print("sentences:", len(sents), "| tokens:", len(tokens), "| vocabulary:", len(set(tokens)))

foll = build(sents, n=2)                 # context (1 word) -> Counter of next words
print("\nwords that most often follow 'the':", foll[("the",)].most_common(6))
print("words that most often follow 'morning':", foll[("morning",)].most_common(5))
print("words that most often start a sentence:", foll[("<s>",)].most_common(5))

sentences: 60 | tokens: 493 | vocabulary: 169

words that most often follow 'the': [('morning', 7), ('weekend', 6), ('park', 5), ('city', 3), ('train', 3), ('rain', 3)]
words that most often follow 'morning': [('</s>', 8), ('train', 2), ('coffee', 2), ('before', 1), ('helps', 1)]
words that most often start a sentence: [('the', 23), ('i', 8), ('we', 8), ('she', 7), ('he', 7)]


In [4]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

sents = to_sentences(open("corpus.txt").read())
foll = build(sents, n=2)

def prob(context_word, word):
    c = foll[(context_word,)]; total = sum(c.values())
    return c[word] / total if total else 0.0

def suggest_next(context_word, k=3):
    c = foll[(context_word,)]; total = sum(c.values())
    return [(w, round(n / total, 3)) for w, n in c.most_common(k)]

print("P(coffee | morning) =", round(prob("morning", "coffee"), 3))
print("P(train  | morning) =", round(prob("morning", "train"), 3))
print()
for w in ["i", "the", "morning", "good", "drink"]:
    print(f"after {w!r:9s}->", suggest_next(w))

P(coffee | morning) = 0.125
P(train  | morning) = 0.125

after 'i'      -> [('drink', 0.125), ('read', 0.125), ('walk', 0.125)]
after 'the'    -> [('morning', 0.103), ('weekend', 0.088), ('park', 0.074)]
after 'morning'-> [('</s>', 0.5), ('train', 0.125), ('coffee', 0.125)]
after 'good'   -> [('book', 0.6), ('coffee', 0.4)]
after 'drink'  -> [('coffee', 0.5), ('tea', 0.5)]


In [ ]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

sents = to_sentences(open("corpus.txt").read())

def generate(foll, n, seed=0, max_len=25):
    rng = random.Random(seed)
    ctx = ("<s>",) * (n - 1); out = []
    for _ in range(max_len):
        c = foll[ctx]
        if not c:
            break
        words, weights = zip(*c.items())
        nxt = rng.choices(words, weights=weights)[0]
        if nxt == "</s>":
            break
        out.append(nxt)
        ctx = (ctx + (nxt,))[-(n - 1):]
    return " ".join(out)

bi, tri = build(sents, 2), build(sents, 3)
print("BIGRAM (1 word of memory):")
for s in range(4):
    print("  -", generate(bi, 2, seed=s))
print("\nTRIGRAM (2 words of memory):")
for s in range(4):
    print("  -", generate(tri, 3, seed=s))

In [5]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

sents = to_sentences(open("corpus.txt").read())
foll = build(sents, n=2)
V = set(t for s in sents for t in to_tokens(s)) | {"</s>"}     # the vocabulary

def prob_mle(ctx, w):
    c = foll[(ctx,)]; total = sum(c.values())
    return c[w] / total if total else 0.0

def prob_addk(ctx, w, k=1.0):                                  # add-k / Laplace smoothing
    c = foll[(ctx,)]; total = sum(c.values())
    return (c[w] + k) / (total + k * len(V))

# 'drinks coffee' never occurs (the corpus has 'drinks his coffee' and 'drinks green tea')
print("MLE   P(coffee | drinks) =", prob_mle("drinks", "coffee"), " <- zero, so 'she drinks coffee' is 'impossible'")
print("add-1 P(coffee | drinks) =", round(prob_addk("drinks", "coffee"), 5), " <- small but non-zero")
print("add-1 P(green  | drinks) =", round(prob_addk("drinks", "green"), 5), " <- a word we DID see still ranks higher")

MLE   P(coffee | drinks) = 0.0  <- zero, so 'she drinks coffee' is 'impossible'
add-1 P(coffee | drinks) = 0.00581  <- small but non-zero
add-1 P(green  | drinks) = 0.01163  <- a word we DID see still ranks higher


In [6]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

sents = to_sentences(open("corpus.txt").read())
rng = random.Random(0); rng.shuffle(sents)
cut = int(0.8 * len(sents)); train, test = sents[:cut], sents[cut:]
V = set(t for s in train for t in to_tokens(s)) | {"</s>", "<unk>"}

def perplexity(foll, n, data, k=1.0):
    log_sum, N = 0.0, 0
    for s in data:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            ctx, w = tuple(toks[i:i + n - 1]), toks[i + n - 1]
            c = foll.get(ctx, Counter()); total = sum(c.values())
            p = (c[w] + k) / (total + k * len(V))              # add-k, so never zero
            log_sum += math.log(p); N += 1
    return math.exp(-log_sum / N)

bi, tri = build(train, 2), build(train, 3)
print(f"bigram  perplexity  ->  train {perplexity(bi, 2, train):6.1f}   held-out {perplexity(bi, 2, test):6.1f}")
print(f"trigram perplexity  ->  train {perplexity(tri, 3, train):6.1f}   held-out {perplexity(tri, 3, test):6.1f}")

bigram  perplexity  ->  train   52.7   held-out   94.1
trigram perplexity  ->  train   60.6   held-out  115.8


In [7]:
import re, math, random
from collections import Counter, defaultdict

def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

class NgramLM:
    "A smoothed n-gram language model: fit -> suggest_next / generate / perplexity."
    def __init__(self, n=2, k=0.5):
        self.n, self.k = n, k

    def fit(self, text):
        self.sents = to_sentences(text)
        self.foll = build(self.sents, self.n)
        self.V = set(t for s in self.sents for t in to_tokens(s)) | {"</s>"}
        return self

    def _prob(self, ctx, w):
        c = self.foll.get(ctx, Counter()); total = sum(c.values())
        return (c[w] + self.k) / (total + self.k * len(self.V))

    def suggest_next(self, text, topk=3):
        ctx = tuple((["<s>"] * (self.n - 1) + to_tokens(text))[-(self.n - 1):])
        c = self.foll.get(ctx, Counter())
        words = [w for w, _ in c.most_common() if w != "</s>"]   # never suggest end-of-sentence
        return words[:topk] or ["(no idea)"]

    def generate(self, seed=0, max_len=25):
        rng = random.Random(seed); ctx = ("<s>",) * (self.n - 1); out = []
        for _ in range(max_len):
            c = self.foll.get(ctx)
            if not c: break
            words, wts = zip(*c.items())
            nxt = rng.choices(words, weights=wts)[0]
            if nxt == "</s>": break
            out.append(nxt); ctx = (ctx + (nxt,))[-(self.n - 1):]
        return " ".join(out)

    def perplexity(self, text):
        log_sum, N = 0.0, 0
        for s in to_sentences(text):
            toks = ["<s>"] * (self.n - 1) + to_tokens(s) + ["</s>"]
            for i in range(len(toks) - self.n + 1):
                ctx, w = tuple(toks[i:i + self.n - 1]), toks[i + self.n - 1]
                log_sum += math.log(self._prob(ctx, w)); N += 1
        return math.exp(-log_sum / N)

lm = NgramLM(n=2, k=0.5).fit(open("corpus.txt").read())
for phrase in ["i drink", "the morning", "on the", "she reads a"]:
    print(f"{phrase!r:15s}->  {lm.suggest_next(phrase)}")
print("\ngenerated:", lm.generate(seed=3))

'i drink'      ->  ['coffee', 'tea']
'the morning'  ->  ['train', 'coffee', 'before']
'on the'       ->  ['morning', 'weekend', 'park']
'she reads a'  ->  ['good', 'book', 'film']

generated: the quiet and coffee
